# LeetCode 1869 — Longer Contiguous Segments of Ones than Zeros bằng Python

Link đề: https://leetcode.com/problems/longer-contiguous-segments-of-ones-than-zeros/

Độ khó: **Easy** · Chủ đề: String

Skeleton chữ ký LeetCode:

```python
class Solution:
    def checkZeroOnes(self, s: str) -> bool:

```

**Mục tiêu học**

1. Hiểu khái niệm **đoạn liên tiếp** (contiguous segment / run) trong chuỗi nhị phân.
2. Ôn duyệt chuỗi, so sánh ký tự với ký tự trước, biến đếm `cur` / kỷ lục `best`, `max()`.
3. Học 4 cách: thử mọi đoạn (brute force) → **một lượt đếm run** → `itertools.groupby` → mẹo `split`.
4. Xử lý trường hợp "không có chữ số 0 (hoặc 1)" ⇒ độ dài 0, và so sánh **nghiêm ngặt** `>`.
5. Trace, chứng minh, Big-O và 6 bài tập về run-length.

## 1. Bài toán yêu cầu gì?

Cho chuỗi nhị phân `s`. Trả `True` nếu **đoạn liên tiếp dài nhất toàn `'1'`** dài **hơn hẳn** (strictly longer) **đoạn liên tiếp dài nhất toàn `'0'`**, ngược lại `False`.

Nếu không có `'0'` nào thì đoạn 0 dài nhất có độ dài 0 (tương tự cho `'1'`).

| s | max run '1' | max run '0' | Kết quả |
|---|---:|---:|:---:|
| `"1101"` | 2 | 1 | `True` |
| `"111000"` | 3 | 3 | `False` (bằng nhau không tính) |
| `"110100010"` | 2 | 3 | `False` |

**Constraints:** `1 <= len(s) <= 100`, `s[i]` là `'0'` hoặc `'1'`.

In [1]:
examples = [("1101", True), ("111000", False), ("110100010", False)]
for s, expected in examples:
    print(repr(s), "->", expected)

'1101' -> True
'111000' -> False
'110100010' -> False


## 2. Nền tảng Python

### 2.1. "Đoạn liên tiếp" (run) là gì?

Một **run** là dãy ký tự **giống nhau, đứng sát nhau**, không thể kéo dài thêm. Chuỗi `"110100010"` chia thành các run: `11 | 0 | 1 | 000 | 1 | 0`.

In [2]:
s = "110100010"
runs = []
start = 0
for i in range(1, len(s) + 1):
    if i == len(s) or s[i] != s[i - 1]:      # hết chuỗi hoặc đổi ký tự -> đóng run
        runs.append(s[start:i])
        start = i
print(" | ".join(runs))
print([(r[0], len(r)) for r in runs])

11 | 0 | 1 | 000 | 1 | 0
[('1', 2), ('0', 1), ('1', 1), ('0', 3), ('1', 1), ('0', 1)]


`s[start:i]` lấy chuỗi con từ `start` tới **trước** `i`. Điều kiện `i == len(s) or ...` đặt `i == len(s)` **trước** để `or` ngắn mạch, không đọc `s[len(s)]` (sẽ lỗi).

### 2.2. So sánh với ký tự trước

Mấu chốt của bài: tại `i`, nếu `s[i] == s[i-1]` thì run hiện tại **dài thêm 1**; nếu khác thì một run mới **bắt đầu** với độ dài 1.

In [3]:
s = "1101"
for i in range(1, len(s)):
    same = s[i] == s[i - 1]
    print(f"i={i}: s[i-1]={s[i-1]} s[i]={s[i]} -> {'nối tiếp run' if same else 'bắt đầu run mới'}")

i=1: s[i-1]=1 s[i]=1 -> nối tiếp run
i=2: s[i-1]=1 s[i]=0 -> bắt đầu run mới
i=3: s[i-1]=0 s[i]=1 -> bắt đầu run mới


### 2.3. Kỷ lục bằng `max()` và dict hai khóa

Ta cần **hai** kỷ lục: cho `'0'` và cho `'1'`. Một dict `best = {'0': 0, '1': 0}` giữ cả hai, tra bằng chính ký tự. Khởi tạo 0 tự động xử lý trường hợp "không có ký tự đó".

In [4]:
best = {"0": 0, "1": 0}
best["1"] = max(best["1"], 2)
best["0"] = max(best["0"], 1)
print(best, "| 1 dài hơn hẳn?", best["1"] > best["0"])

{'0': 1, '1': 2} | 1 dài hơn hẳn? True


### 2.4. Chuỗi con kiểm tra "toàn một ký tự"

`set(t) == {'1'}` kiểm `t` khác rỗng và chỉ gồm `'1'`. Hoặc `t == '1' * len(t)`. Dùng trong cách brute force.

In [5]:
for t in ["111", "101", ""]:
    print(repr(t), set(t) == {"1"}, t == "1" * len(t))

'111' True True
'101' False False
'' False True


Chú ý chuỗi rỗng: `"" == "1" * 0` là `True` — cách thứ hai coi chuỗi rỗng là "toàn 1". Brute force cần chỉ xét chuỗi con **khác rỗng**.

## 3. Cách 1 — brute force: thử mọi đoạn

Với mọi cặp `i <= j`, nếu `s[i..j]` toàn `'1'` thì cập nhật max run 1; toàn `'0'` thì cập nhật max run 0. Đây là cách hiểu đề trực tiếp nhất.

In [6]:
def check_zero_ones_bruteforce(s: str) -> bool:
    n = len(s)
    best1 = best0 = 0
    for i in range(n):
        for j in range(i, n):
            seg = s[i:j + 1]
            if set(seg) == {"1"}:
                best1 = max(best1, len(seg))
            elif set(seg) == {"0"}:
                best0 = max(best0, len(seg))
    return best1 > best0

for s, expected in examples:
    print(s, "->", check_zero_ones_bruteforce(s), "| expected", expected)
    assert check_zero_ones_bruteforce(s) == expected

1101 -> True | expected True
111000 -> False | expected False
110100010 -> False | expected False


### 3.1. Nút thắt

Có `n(n+1)/2` đoạn, mỗi đoạn tốn `O(n)` để cắt và tạo `set` ⇒ `O(n³)`. Với `n = 100` khoảng 170.000 thao tác — vẫn chạy, nhưng **lãng phí**: một đoạn toàn `'1'` dài 5 bị kiểm lại nhiều lần qua mọi đoạn con của nó.

Cải tiến nhỏ: cố định `i`, kéo dài `j` cho tới khi gặp ký tự khác rồi **dừng** — `O(n²)`.

In [7]:
def check_zero_ones_extend(s: str) -> bool:
    best = {"0": 0, "1": 0}
    n = len(s)
    for i in range(n):
        j = i
        while j < n and s[j] == s[i]:
            j += 1
        best[s[i]] = max(best[s[i]], j - i)
    return best["1"] > best["0"]

for s, expected in examples:
    assert check_zero_ones_extend(s) == expected
print("O(n^2) kéo dài OK")

O(n^2) kéo dài OK


Vẫn còn lặp: từ `i` và từ `i+1` cùng trong một run sẽ quét lại gần như cùng đoạn. **Insight:** chỉ cần **một lượt** — mỗi ký tự hoặc nối dài run hiện tại, hoặc mở run mới.

### 3.2. Insight — đếm run trong một lượt

Giữ `cur` = độ dài run **đang kết thúc tại `i`**:

- `s[i] == s[i-1]` ⇒ `cur += 1`.
- Khác ⇒ `cur = 1` (run mới chỉ có `s[i]`).

Sau mỗi bước, cập nhật kỷ lục của **ký tự `s[i]`**: `best[s[i]] = max(best[s[i]], cur)`.

Không cần biết run kết thúc ở đâu — vì ta cập nhật kỷ lục **ở mọi bước**, nên khi run dài nhất đạt độ dài đầy đủ, kỷ lục đã được ghi.

In [8]:
s = "110100010"
cur = 0
for i, ch in enumerate(s):
    cur = cur + 1 if i > 0 and ch == s[i - 1] else 1
    print(f"i={i} ch={ch} cur={cur}")

i=0 ch=1 cur=1
i=1 ch=1 cur=2
i=2 ch=0 cur=1
i=3 ch=1 cur=1
i=4 ch=0 cur=1
i=5 ch=0 cur=2
i=6 ch=0 cur=3
i=7 ch=1 cur=1
i=8 ch=0 cur=1


Dòng `cur = cur + 1 if điều_kiện else 1` là **biểu thức điều kiện** (ternary): chọn `cur + 1` hoặc `1`.

## 4. Lời giải chính — giảng từng đoạn

**Đoạn 1:** `best = {'0': 0, '1': 0}`, `cur = 0`, `prev = ''` (chưa có ký tự trước).

**Đoạn 2:** duyệt `ch in s`: nếu `ch == prev` thì `cur += 1`, ngược lại `cur = 1`.

**Đoạn 3:** `best[ch] = max(best[ch], cur)`; `prev = ch`.

**Đoạn 4:** `return best['1'] > best['0']`.

Dùng `prev = ''` khiến ký tự đầu tiên luôn "khác prev" ⇒ `cur = 1` — không cần xử lý riêng `i = 0`.

In [9]:
def check_zero_ones(s: str) -> bool:
    best = {"0": 0, "1": 0}
    cur = 0
    prev = ""
    for ch in s:
        if ch == prev:
            cur += 1
        else:
            cur = 1
        best[ch] = max(best[ch], cur)
        prev = ch
    return best["1"] > best["0"]

for s, expected in examples:
    assert check_zero_ones(s) == expected
print([check_zero_ones(s) for s, _ in examples])

[True, False, False]


## 5. TRACE từng bước

Trace `s = "110100010"`:

| i | ch | prev | cur | best['0'] | best['1'] |
|---:|:---:|:---:|---:|---:|---:|
| 0 | 1 | '' | 1 | 0 | 1 |
| 1 | 1 | 1 | 2 | 0 | 2 |
| 2 | 0 | 1 | 1 | 1 | 2 |
| 3 | 1 | 0 | 1 | 1 | 2 |
| 4 | 0 | 1 | 1 | 1 | 2 |
| 5 | 0 | 0 | 2 | 2 | 2 |
| 6 | 0 | 0 | 3 | 3 | 2 |
| 7 | 1 | 0 | 1 | 3 | 2 |
| 8 | 0 | 1 | 1 | 3 | 2 |

Kết thúc: `2 > 3`? **Không** ⇒ `False`.

In [10]:
def trace_check_zero_ones(s: str) -> bool:
    best = {"0": 0, "1": 0}
    cur, prev = 0, ""
    print(" i | ch | prev | cur | best0 | best1 | ghi chú")
    for i, ch in enumerate(s):
        note = "nối run" if ch == prev else "run mới"
        cur = cur + 1 if ch == prev else 1
        old = best[ch]
        best[ch] = max(best[ch], cur)
        if best[ch] > old:
            note += f", kỷ lục '{ch}' = {best[ch]}"
        print(f"{i:>2} |  {ch} | {prev!r:>4} | {cur:>3} | {best['0']:>5} | {best['1']:>5} | {note}")
        prev = ch
    res = best["1"] > best["0"]
    print(f"=> {best['1']} > {best['0']} ? {res}")
    return res

assert trace_check_zero_ones("110100010") is False

 i | ch | prev | cur | best0 | best1 | ghi chú
 0 |  1 |   '' |   1 |     0 |     1 | run mới, kỷ lục '1' = 1
 1 |  1 |  '1' |   2 |     0 |     2 | nối run, kỷ lục '1' = 2
 2 |  0 |  '1' |   1 |     1 |     2 | run mới, kỷ lục '0' = 1
 3 |  1 |  '0' |   1 |     1 |     2 | run mới
 4 |  0 |  '1' |   1 |     1 |     2 | run mới
 5 |  0 |  '0' |   2 |     2 |     2 | nối run, kỷ lục '0' = 2
 6 |  0 |  '0' |   3 |     3 |     2 | nối run, kỷ lục '0' = 3
 7 |  1 |  '0' |   1 |     3 |     2 | run mới
 8 |  0 |  '1' |   1 |     3 |     2 | run mới
=> 2 > 3 ? False


Trace thứ hai: chuỗi **không có `'0'`** — `best['0']` giữ nguyên 0 nhờ khởi tạo.

In [11]:
assert trace_check_zero_ones("111") is True
print()
assert trace_check_zero_ones("0") is False

 i | ch | prev | cur | best0 | best1 | ghi chú
 0 |  1 |   '' |   1 |     0 |     1 | run mới, kỷ lục '1' = 1
 1 |  1 |  '1' |   2 |     0 |     2 | nối run, kỷ lục '1' = 2
 2 |  1 |  '1' |   3 |     0 |     3 | nối run, kỷ lục '1' = 3
=> 3 > 0 ? True

 i | ch | prev | cur | best0 | best1 | ghi chú
 0 |  0 |   '' |   1 |     1 |     0 | run mới, kỷ lục '0' = 1
=> 0 > 1 ? False


## 6. Cách 3 — `itertools.groupby`

`groupby(s)` gom các ký tự **liền kề giống nhau** thành nhóm, trả cặp `(ký_tự, iterator_nhóm)`. Đây chính là tách run có sẵn trong thư viện chuẩn.

In [12]:
from itertools import groupby

for key, group in groupby("110100010"):
    g = list(group)
    print(key, g, "độ dài", len(g))

1 ['1', '1'] độ dài 2
0 ['0'] độ dài 1
1 ['1'] độ dài 1
0 ['0', '0', '0'] độ dài 3
1 ['1'] độ dài 1
0 ['0'] độ dài 1


In [13]:
def check_zero_ones_groupby(s: str) -> bool:
    best = {"0": 0, "1": 0}
    for key, group in groupby(s):
        best[key] = max(best[key], len(list(group)))
    return best["1"] > best["0"]

for s, expected in examples:
    assert check_zero_ones_groupby(s) == expected
print("groupby OK")

groupby OK


Lưu ý: `groupby` chỉ gom phần tử **liền kề** — khác với "nhóm theo giá trị" của SQL. Đó đúng là điều ta cần.

## 7. Cách 4 — mẹo `split`

`s.split('0')` cắt chuỗi tại **mỗi** ký tự `'0'`, các mảnh còn lại là các đoạn **toàn `'1'`** (có thể rỗng). Độ dài mảnh dài nhất = max run của `'1'`. Tương tự `s.split('1')` cho `'0'`.

In [14]:
s = "110100010"
print("split('0'):", s.split("0"))
print("split('1'):", s.split("1"))
print("max run 1 =", max(map(len, s.split("0"))), "| max run 0 =", max(map(len, s.split("1"))))

split('0'): ['11', '1', '', '', '1', '']
split('1'): ['', '', '0', '000', '0']
max run 1 = 2 | max run 0 = 3


In [15]:
def check_zero_ones_split(s: str) -> bool:
    return max(map(len, s.split("0"))) > max(map(len, s.split("1")))

for s, expected in examples:
    assert check_zero_ones_split(s) == expected
print(check_zero_ones_split("111"), "| '111'.split('1') =", "111".split("1"))

True | '111'.split('1') = ['', '', '', '']


Khi không có `'0'`, `"111".split("1")` cho `['', '', '', '']` — các mảnh rỗng có độ dài 0 ⇒ max run 0 = 0. Mẹo này tự xử lý biên! `map(len, list)` áp `len` lên từng phần tử.

## 8. Code nộp LeetCode

### Cách A — một lượt đếm run (khuyên dùng)

In [16]:
class Solution:
    def checkZeroOnes(self, s: str) -> bool:
        best = {"0": 0, "1": 0}
        cur, prev = 0, ""
        for ch in s:
            cur = cur + 1 if ch == prev else 1
            best[ch] = max(best[ch], cur)
            prev = ch
        return best["1"] > best["0"]

print([Solution().checkZeroOnes(s) for s, _ in examples])

[True, False, False]


### Cách B — `split` (một dòng)

In [17]:
class SolutionSplit:
    def checkZeroOnes(self, s: str) -> bool:
        return max(map(len, s.split("0"))) > max(map(len, s.split("1")))

print([SolutionSplit().checkZeroOnes(s) for s, _ in examples])

[True, False, False]


### Cách C — `groupby`

In [18]:
class SolutionGroupby:
    def checkZeroOnes(self, s: str) -> bool:
        best = {"0": 0, "1": 0}
        for key, grp in groupby(s):
            best[key] = max(best[key], sum(1 for _ in grp))
        return best["1"] > best["0"]

print([SolutionGroupby().checkZeroOnes(s) for s, _ in examples])

[True, False, False]


`sum(1 for _ in grp)` đếm số phần tử của iterator mà không cần tạo list.

## 9. Vì sao đúng?

**Invariant:** sau khi xử lý `s[0..i]`:

1. `cur` = độ dài run chứa `s[i]` tính tới `i` (tức số ký tự giống `s[i]` liên tiếp kết thúc tại `i`).
2. `best[c]` = độ dài lớn nhất của các đoạn toàn `c` **nằm trong** `s[0..i]`.

- (1) đúng: nếu `s[i] == s[i-1]`, run tại `i` là run tại `i-1` kéo thêm 1; nếu khác, run tại `i` chỉ có `s[i]`.
- (2) đúng: mọi đoạn toàn `c` trong `s[0..i]` mà kết thúc tại `i` có độ dài ≤ `cur`, và đoạn dài nhất kết thúc tại `i` có đúng độ dài `cur`; các đoạn kết thúc trước đã được tính ở bước trước.

Kết thúc: `best` là max run thật sự của mỗi ký tự; so sánh `>` đúng yêu cầu "dài hơn hẳn".

## 10. Độ phức tạp

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| Brute force mọi đoạn | `O(n³)` | `O(n)` (chuỗi con) |
| Kéo dài từ mỗi `i` | `O(n²)` | `O(1)` |
| Một lượt đếm run | `O(n)` | `O(1)` |
| `groupby` | `O(n)` | `O(1)` (bản `sum`) |
| `split` | `O(n)` | `O(n)` (list mảnh) |

In [19]:
import time
s_big = "10" * 50
for f in [check_zero_ones_bruteforce, check_zero_ones_extend, check_zero_ones, check_zero_ones_split]:
    t = time.perf_counter()
    for _ in range(20):
        f(s_big)
    print(f"{f.__name__:<28} {(time.perf_counter() - t) * 1000:8.2f} ms / 20 lần (tùy máy)")

check_zero_ones_bruteforce      59.72 ms / 20 lần (tùy máy)
check_zero_ones_extend           0.28 ms / 20 lần (tùy máy)
check_zero_ones                  0.14 ms / 20 lần (tùy máy)
check_zero_ones_split            0.06 ms / 20 lần (tùy máy)


## 11. Bộ test

Nhóm: mẫu đề; một ký tự `'1'` / `'0'`; toàn `'1'`; toàn `'0'`; bằng nhau; run dài nhất ở **cuối** chuỗi (dễ quên cập nhật nếu chỉ cập nhật khi đổi ký tự); ngẫu nhiên đối chiếu.

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS ({actual})")

sol = Solution()
check("mẫu 1", sol.checkZeroOnes("1101"), True)
check("mẫu 2", sol.checkZeroOnes("111000"), False)
check("mẫu 3", sol.checkZeroOnes("110100010"), False)
check("'1'", sol.checkZeroOnes("1"), True)
check("'0'", sol.checkZeroOnes("0"), False)
check("toàn 1", sol.checkZeroOnes("1" * 100), True)
check("xen kẽ 1010", sol.checkZeroOnes("1010"), False)
check("run 1 dài nhất ở cuối", sol.checkZeroOnes("0010111"), True)

mẫu 1                    PASS (True)
mẫu 2                    PASS (False)
mẫu 3                    PASS (False)
'1'                      PASS (True)
'0'                      PASS (False)
toàn 1                   PASS (True)
xen kẽ 1010              PASS (False)
run 1 dài nhất ở cuối    PASS (True)


In [21]:
import random
rng = random.Random(1869)
impls = [check_zero_ones_bruteforce, check_zero_ones_extend, check_zero_ones, check_zero_ones_groupby,
         check_zero_ones_split, Solution().checkZeroOnes, SolutionSplit().checkZeroOnes,
         SolutionGroupby().checkZeroOnes]
for _ in range(1500):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 40)))
    res = {f(s) for f in impls}
    assert len(res) == 1, (s, res)
print("8 cài đặt đồng ý trên 1500 chuỗi ngẫu nhiên PASS")

8 cài đặt đồng ý trên 1500 chuỗi ngẫu nhiên PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: chỉ cập nhật kỷ lục khi **đổi** ký tự ⇒ quên run cuối

In [22]:
def wrong_update_on_change(s):
    best = {"0": 0, "1": 0}
    cur = 1
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            cur += 1
        else:
            best[s[i - 1]] = max(best[s[i - 1]], cur)    # chỉ ghi khi run kết thúc
            cur = 1
    return best["1"] > best["0"]                          # quên ghi run cuối!

print("sai :", wrong_update_on_change("0111"), "| đúng:", check_zero_ones("0111"))

sai : False | đúng: True


Sửa: cập nhật kỷ lục **mỗi bước**, hoặc thêm một lần cập nhật sau vòng lặp.

### Lỗi 2: dùng `>=` thay vì `>`

In [23]:
def wrong_ge(s):
    return max(map(len, s.split("0"))) >= max(map(len, s.split("1")))

print("sai :", wrong_ge("111000"), "| đúng:", check_zero_ones("111000"))

sai : True | đúng: False


Đề: "strictly longer" ⇒ `>`.

### Lỗi 3: đếm **tổng** số `'1'` thay vì run dài nhất

In [24]:
def wrong_total_count(s):
    return s.count("1") > s.count("0")

print("sai :", wrong_total_count("00011101011"), "| đúng:", check_zero_ones("00011101011"), "(6 số 1 > 5 số 0, nhưng run dài nhất đều là 3)")
print("ví dụ đề '110100010':", wrong_total_count("110100010"), check_zero_ones("110100010"))

sai : True | đúng: False (6 số 1 > 5 số 0, nhưng run dài nhất đều là 3)
ví dụ đề '110100010': False False


`s.count('1')` đếm **mọi** `'1'`, không quan tâm liền kề. Đề hỏi về **đoạn liên tiếp**.

### Lỗi 4: quên reset `cur = 1` khi đổi ký tự

In [25]:
def wrong_no_reset(s):
    best = {"0": 0, "1": 0}
    cur, prev = 0, ""
    for ch in s:
        cur += 1 if ch == prev else 0      # không reset về 1!
        best[ch] = max(best[ch], cur)
        prev = ch
    return best["1"] > best["0"]

print("sai :", wrong_no_reset("1101"), "| đúng:", check_zero_ones("1101"))

sai : False | đúng: True


`cur` không bao giờ quay về 1 ⇒ độ dài bị cộng dồn qua các run khác nhau (và ký tự đầu có `cur = 0`!). Sửa: `cur = 1` khi gặp ký tự mới.

## 13. Mẹo nhận dạng pattern

- "Đoạn **liên tiếp** giống nhau dài nhất" ⇒ **run-length counting**: `cur` (độ dài run hiện tại) + `best` (kỷ lục), so với ký tự trước.
- Cần tách run thành nhóm ⇒ `itertools.groupby`.
- Chuỗi chỉ có 2 loại ký tự ⇒ mẹo `split` theo ký tự kia.
- Luôn kiểm: **run cuối cùng** có được tính không? Biên "không có ký tự nào" có ra 0 không?
- "Dài hơn hẳn" ⇒ `>`; "ít nhất bằng" ⇒ `>=`.

## 14. Biến thể

1. **LeetCode 485 — Max Consecutive Ones**: chỉ cần max run của `1` trong mảng.
2. **LeetCode 1446 — Consecutive Characters**: max run của **bất kỳ** ký tự nào.
3. **Run-length encoding**: `"aaabcc"` → `"a3b1c2"`.
4. **LeetCode 1004 — Max Consecutive Ones III**: được lật tối đa `k` số 0 ⇒ sliding window (khó hơn).

In [26]:
def max_consecutive_ones(nums):
    best = cur = 0
    for x in nums:
        cur = cur + 1 if x == 1 else 0
        best = max(best, cur)
    return best

def rle(s):
    return "".join(f"{k}{len(list(g))}" for k, g in groupby(s))

print(max_consecutive_ones([1, 1, 0, 1, 1, 1]))
print(rle("aaabcc"))

3
a3b1c2


## 15. Bài tập (tự làm trước)

**Bài 1 — tính tay:** với `s = "0011100110000"`, max run 1 và max run 0 là bao nhiêu? Kết quả?

**Bài 2 — dự đoán output:**

```python
print("1001".split("0"), "1001".split("1"))
print([len(list(g)) for _, g in groupby("aabbba")])
```

**Bài 3 — điền TODO:** một lượt đếm run.

In [27]:
def exercise_check_zero_ones(s):
    # TODO 1: best = {"0": 0, "1": 0}; cur = 0; prev = ""
    # TODO 2: ch == prev -> cur += 1, ngược lại cur = 1
    # TODO 3: cập nhật best[ch]; prev = ch
    # TODO 4: return best["1"] > best["0"]
    pass

# assert exercise_check_zero_ones("1101") is True
# assert exercise_check_zero_ones("111000") is False

**Bài 4 — LeetCode 1446:** trả về độ dài run dài nhất của **bất kỳ** ký tự nào trong chuỗi `s` (không rỗng).

In [28]:
def exercise_max_power(s):
    # TODO: như bài chính nhưng chỉ một kỷ lục chung
    pass

# assert exercise_max_power("leetcode") == 2
# assert exercise_max_power("abbcccddddeeeeedcba") == 5

**Bài 5 — run-length encoding:** `exercise_rle("aaabcc") == "a3b1c2"`. Không dùng `groupby`.

In [29]:
def exercise_rle(s):
    # TODO: duyệt, khi đổi ký tự (hoặc hết chuỗi) thì ghi ký tự + độ dài run
    pass

# assert exercise_rle("aaabcc") == "a3b1c2"
# assert exercise_rle("z") == "z1"

**Bài 6 — mini challenge (LC 1004):** mảng nhị phân `nums` và số `k`; được lật tối đa `k` số `0` thành `1`. Trả về độ dài đoạn liên tiếp toàn `1` dài nhất có thể.

In [30]:
def exercise_longest_ones_k(nums, k):
    # TODO: sliding window: đếm số 0 trong cửa sổ, nếu > k thì co trái
    pass

# assert exercise_longest_ones_k([1,1,1,0,0,0,1,1,1,1,0], 2) == 6
# assert exercise_longest_ones_k([0,0,1,1,0,0,1,1,1,0,1,1,0,0,0,1,1,1,1], 3) == 10

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | tách run | `00|111|00|11|0000` | 3 vs 4 ⇒ `False` |
| 2 | split cắt tại ký tự | mảnh rỗng giữa hai `0` | groupby gom liền kề |
| 3 | `prev = ""` | ternary `cur + 1 if ... else 1` | `best[ch] = max(best[ch], cur)` |
| 4 | một `best` | so với `s[i-1]` | cập nhật mỗi bước |
| 5 | `cur` và `prev` | ghi khi đổi ký tự | nhớ ghi run cuối sau vòng lặp |
| 6 | cửa sổ `[left, right]` | `zeros` đếm số 0 | `while zeros > k: ...; left += 1` |

## 17. Đáp án tham khảo

**Bài 1:** runs `00 | 111 | 00 | 11 | 0000` ⇒ max 1 = 3, max 0 = 4 ⇒ `False`.

**Bài 2:** `['1', '', '1'] ['', '00', '']` và `[2, 3, 1]`.

In [31]:
print(check_zero_ones("0011100110000"))
print("1001".split("0"), "1001".split("1"))
print([len(list(g)) for _, g in groupby("aabbba")])

False
['1', '', '1'] ['', '00', '']
[2, 3, 1]


In [32]:
def answer_check_zero_ones(s):
    best = {"0": 0, "1": 0}
    cur, prev = 0, ""
    for ch in s:
        cur = cur + 1 if ch == prev else 1
        best[ch] = max(best[ch], cur)
        prev = ch
    return best["1"] > best["0"]


def answer_max_power(s):
    best = cur = 1
    for i in range(1, len(s)):
        cur = cur + 1 if s[i] == s[i - 1] else 1
        best = max(best, cur)
    return best


def answer_rle(s):
    out = []
    cur = 1
    for i in range(1, len(s) + 1):
        if i < len(s) and s[i] == s[i - 1]:
            cur += 1
        else:
            out.append(f"{s[i - 1]}{cur}")
            cur = 1
    return "".join(out)


def answer_longest_ones_k(nums, k):
    left = zeros = best = 0
    for right, x in enumerate(nums):
        if x == 0:
            zeros += 1
        while zeros > k:
            if nums[left] == 0:
                zeros -= 1
            left += 1
        best = max(best, right - left + 1)
    return best


assert answer_check_zero_ones("1101") is True
assert answer_check_zero_ones("111000") is False
assert answer_max_power("leetcode") == 2
assert answer_max_power("abbcccddddeeeeedcba") == 5
assert answer_rle("aaabcc") == "a3b1c2"
assert answer_rle("z") == "z1"
assert answer_longest_ones_k([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2) == 6
assert answer_longest_ones_k([0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1], 3) == 10
for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 20)))
    assert answer_rle(s) == rle(s)
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def checkZeroOnes(self, s: str) -> bool:
        best = {"0": 0, "1": 0}
        cur, prev = 0, ""
        for ch in s:
            cur = cur + 1 if ch == prev else 1     # nối run hoặc mở run mới
            best[ch] = max(best[ch], cur)          # cập nhật MỌI bước
            prev = ch
        return best["1"] > best["0"]               # dài hơn HẲN
```

Khung run-length: **`cur` = run hiện tại, `best` = kỷ lục, so với ký tự trước.**

Checklist:

- [ ] Cập nhật kỷ lục ở mọi bước (hoặc nhớ xử lý run cuối).
- [ ] Reset `cur = 1` khi đổi ký tự.
- [ ] Khởi tạo kỷ lục bằng 0 cho trường hợp không có ký tự đó.
- [ ] So sánh `>` (strictly).
- [ ] Đếm run **liên tiếp**, không phải tổng số lần xuất hiện.